In [ ]:
!pip install -q torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers ftfy regex pyyaml "numpy<2"

In [ ]:
import os
import base64
import zipfile
import sys

os.makedirs('/kaggle/working/ACHG-CLIP', exist_ok=True)
os.chdir('/kaggle/working/ACHG-CLIP')

zip_b64 = "..."

with open('code.zip', 'wb') as f:
    f.write(base64.b64decode(zip_b64))

with zipfile.ZipFile('code.zip', 'r') as z:
    z.extractall('.')

print('Codebase successfully extracted!')
print('Contents of current directory:', os.listdir('.'))
if os.path.exists('data'):
    print('Contents of data:', os.listdir('data'))
else:
    print('DATA FOLDER IS MISSING!')


In [ ]:

import os
os.chdir('/kaggle/working/ACHG-CLIP')

import sys
os.makedirs('scratch', exist_ok=True)
with open('scratch/run_ablation.py', 'w') as f:
    f.write('import sys\nimport yaml\nfrom unittest.mock import patch\nimport torch\n\n# Add parent directory to path so we can import from the codebase\nimport os\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \'..\')))\n\nimport run_cifar100\nimport models.achg_clip\n\nclass NoOpFeedbackPath(models.achg_clip.FeedbackPath):\n    """\n    Bypasses the feedback loop, meaning q_final is NOT injected back into the prompt.\n    The graph branch executes, but the prompts are only updated via the contrastive loss.\n    """\n    def forward(self, hgnec_output, prompt_injector):\n        return {\n            "status": "DISABLED",\n            "q_reshaped": None,\n            "q_final": hgnec_output.q_final,\n            "applied": False,\n        }\n\noriginal_safe_load = yaml.safe_load\n\ndef patched_safe_load(stream):\n    data = original_safe_load(stream)\n    if isinstance(data, dict) and "lambda_recon" in data:\n        print(f"\\n[ABLATION RUNNER] Intercepting training.yaml")\n        if "--no_acga" in sys.argv:\n            print("[ABLATION RUNNER] Applying No ACGA ablation (lambda_recon=0, lambda_adv=0)")\n            data["lambda_recon"]["value"] = 0.0\n            data["lambda_adv"]["value"] = 0.0\n        if "--prompt_only" in sys.argv:\n            print("[ABLATION RUNNER] Applying Prompt-Only ablation (lambda_*=0)")\n            data["lambda_recon"]["value"] = 0.0\n            data["lambda_adv"]["value"] = 0.0\n            data["lambda_energy"]["value"] = 0.0\n    return data\n\nif __name__ == "__main__":\n    if "--prompt_only" in sys.argv:\n        print("[ABLATION RUNNER] Replacing ResolvedFeedbackPath with NoOpFeedbackPath")\n        models.achg_clip.ResolvedFeedbackPath = NoOpFeedbackPath\n        sys.argv.remove("--prompt_only")\n    \n    if "--no_acga" in sys.argv:\n        sys.argv.remove("--no_acga")\n        \n    with patch(\'yaml.safe_load\', side_effect=patched_safe_load):\n        run_cifar100.main()\n')

print("=== STARTING ABLATION 1: NO ACGA ===")
!PYTHONPATH=/kaggle/working/ACHG-CLIP python scratch/run_ablation.py --variant ViT-L/14 --seed 42 --no_acga

print("=== STARTING ABLATION 2: PROMPT ONLY ===")
!PYTHONPATH=/kaggle/working/ACHG-CLIP python scratch/run_ablation.py --variant ViT-L/14 --seed 42 --prompt_only

